# standardiser alt input til model
    - [x] Domæne
    - [x] Data transformation
    - Fiks har constraint
    - undersøg stabil træning
    - Fiks data loader så den returnere abselut z til fourer embedding

In [1]:
import sys
import torch

import neuralop as nop

from pathlib import Path
from neuralop.models.base_model import BaseModel

from torch.utils.data import DataLoader, Dataset, Subset


ROOT_DIR = Path().resolve().parents[1]
sys.path.insert(0, str(ROOT_DIR))

from hesel_scraper.bout_dump import BOUTHESELInfo
from hesel_scraper.bout_phys import BOUTHESELPhys

root = ROOT_DIR / r"sim_data/data_25_512_Alexander"
info = BOUTHESELInfo(root)
phys = BOUTHESELPhys(info)

In [6]:
# wrapper

class WrappedFNO(BaseModel):

    def __init__(self, fno, info, phys, m: int = 3):
        super().__init__()

        self.m = m # z width
        self.info = info

        self.dtype = self.info.dtype
        self.device = self.info.device

        self.fno = fno.to(self.device)

        # Til koordinat håndtering
        self.nx = self.info.parameters.num_x
        self.lx = self.info.parameters.Lx
        self.dz = self.info.parameters.dz

        self.xline = torch.linspace(0, self.lx, self.nx, device=self.device, dtype=self.dtype)
        self.z_offset = torch.linspace(-self.dz/2, self.dz/2, self.m, device=self.device, dtype=self.dtype)  # [m]

        # Til state håndtering
        self.register_buffer("mean", self.info.standardized.mean.view(1, 1, 1, -1))
        self.register_buffer("std", self.info.standardized.std.view(1, 1, 1, -1))



    def _cord_embedding(self, z_center: torch.Tensor, t_coord: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Embedder de fysiske koordinater x, z og t
        x (Linear standardisering): [0, Lx] -> [-1, 1]:     x' = 2 * x / Lx - 1
        z (Fourier embedding):      [0, Lz] -> [z_1, z_2]:  z' = [cos(2 * pi * z / Lz), sin(2 * pi * z / Lz)] min max: (-1, 1)
        t (Linear standardisering): [0, Lt] -> [0, 1]:      t' = t / Lt

            Input:
                z: Tensor of shape (batch_size,) representing the z mid coordinate.
                t: Tensor of shape (batch_size,) representing the time coordinate.
        """

        B = z_center.shape[0]


        z_line = z_center[:, None] + self.z_offset[None, :]  # [B,m]

        x_phys = self.xline[None, :, None].expand(B, self.nx, self.m)
        z_phys = z_line[:, None, :].expand(B, self.nx, self.m)
        t_phys = t_coord[:, None, None].expand(B, self.nx, self.m)

        x_phys.requires_grad_(True)
        z_phys.requires_grad_(True)
        t_phys.requires_grad_(True)

        x_hat = 2 * x_phys / self.lx - 1
        z_1 = torch.cos(2 * torch.pi * z_phys / self.info.parameters.Lz)
        z_2 = torch.sin(2 * torch.pi * z_phys / self.info.parameters.Lz)
        t_hat = t_phys / self.info.parameters.Lt

        coord_emb = torch.stack([x_hat, z_1, z_2, t_hat], dim=-1)   # [B, nx, m, 4]
        coord_phys = torch.stack([x_phys, z_phys, t_phys], dim=-1)  # [B, nx, m, 3]

        return coord_emb, coord_phys
    
    def _u_standardize(self, u):
        """
        Standardiserer u
        """
        return (u - self.mean) / self.std
    

    def forward(self, batch):
        u, _, t, z, _ = batch

        u_std = self._u_standardize(u)
        coord_emb, coord_phys = self._cord_embedding(z, t)

        model_input = torch.cat([u_std, coord_emb], dim=-1)
        model_input = model_input.permute(0, 3, 1, 2)

        f_std = self.fno(model_input)
        f = f_std.permute(0, 2, 3, 1)

        f = f * self.std + self.mean

        f_theta = {
            "lnn": f[..., 0:1],
            "lnpe": f[..., 1:2],
            "lnpi": f[..., 2:3],
            "phi": f[..., 3:4],
        }

        return f_theta, coord_phys

In [19]:
# data loder

# skal returnere, u[t-1], cord_phys, avg_z[x], cord_num

import torch
from torch.utils.data import Dataset


class HESEL_z_Dataset(Dataset):
    def __init__(self, info, m: int = 3):
        super().__init__()

        self.info = info
        self.m = m
        self.half = m // 2

        self.dtype = info.dtype

        self.nz = info.parameters.num_z
        self.nt = info.parameters.num_t

        self.dz = info.parameters.dz
        self.dt = info.parameters.dt

        # [nt, 4, nx, nz]
        self.data = torch.stack([info.data.lnn,
                                 info.data.lnpe,
                                 info.data.lnpi,
                                 info.data.phi], dim=1)
        
        # [nt, nx, 4]
        self.avg_z = torch.stack((self.info.avg_in_z["avg_n"].squeeze(-1),
                                  self.info.avg_in_z["avg_te"].squeeze(-1),
                                  self.info.avg_in_z["avg_ti"].squeeze(-1),
                                  self.info.avg_in_z["avg_phi"].squeeze(-1)), dim=-1)

        self.z_offsets = torch.arange(-self.half, self.half + 1)  # [m]



    def __len__(self):
        return (self.nt - 1) * self.nz

    def _get_u_window(self, t_idx: int, z_idx: int):
        z_indices = (z_idx + self.z_offsets) % self.nz
        
        # [4, nx, m]
        u = self.data[t_idx, :, :, z_indices]

        # [nx, m, 4]
        u = u.permute(1, 2, 0)
        return u

    def __getitem__(self, idx):
        t_idx = idx // self.nz
        z_idx = idx % self.nz

        t_phys = torch.tensor(self.dt * t_idx, dtype=self.dtype)
        z_phys = torch.tensor(self.dz * z_idx, dtype=self.dtype)

        cord_phys = torch.tensor([z_phys, t_phys], dtype=self.dtype)

        u = self._get_u_window(t_idx, z_idx)
        y = self._get_u_window(t_idx + 1, z_idx)

        avg_z = self.avg_z[t_idx, :, :].unsqueeze(1).expand(-1, self.m, -1)  # [nx, m, 4]

        return u, y, cord_phys, avg_z

In [21]:
from torch.utils.data import DataLoader
m = 3

dataset = HESEL_z_Dataset(info, m=m)

loader = DataLoader(
    dataset,
    batch_size=10,
    shuffle=True,
)

batch = next(iter(loader))
batch = tuple(b.to(device=info.device) for b in batch)


# cord_num = batch[3]

# fno = nop.models.FNO(n_modes=(160, 160),
#                         in_channels=8,
#                         out_channels=4,
#                         hidden_channels=20,
#                         positional_embedding=None)

# model = WrappedFNO(fno, info, phys, m)

# predicted, cord_phys = model.forward(batch)


